# ORION ENTERPRISES
# Solar Distribution Operations
# Data Cleaning & Preparation

In [2]:
import pandas as pd
import numpy as np

# 1. LOAD DATASET

In [3]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Orion Enterprises/orion_solar_operations.csv")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (3900, 29)


# 2. INITIAL DATA INSPECTION

In [4]:
print("\nFirst 5 rows:")
print(df.head())


First 5 rows:
  transaction_id transaction_date    customer_id customer_type customer_city  \
0  ORN-TXN-00001       2025-08-03  ORN-CUST-0705        Dealer      Yavatmal   
1  ORN-TXN-00002       2025-09-02  ORN-CUST-0806        Dealer      Yavatmal   
2  ORN-TXN-00003       2025-06-16  ORN-CUST-0811     Installer      Amravati   
3  ORN-TXN-00004       2025-04-13  ORN-CUST-0638     Installer        Nagpur   
4  ORN-TXN-00005       2025-04-09  ORN-CUST-0357        Dealer      Amravati   

           sales_region         sales_channel    product_category  \
0         Vidarbha Core            B2B Dealer      Electrical BOS   
1         Vidarbha Core            B2B Dealer      Electrical BOS   
2         Vidarbha Core  Rooftop Installation      Electrical BOS   
3  Maharashtra Extended  Rooftop Installation      Electrical BOS   
4         Vidarbha Core            B2B Dealer  Mounting Structure   

                    product_name  brand  ... payment_method  payment_status  \
0         

In [5]:

print("\nDataset information:")
print(df.info())


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   transaction_id               3900 non-null   object 
 1   transaction_date             3900 non-null   object 
 2   customer_id                  3900 non-null   object 
 3   customer_type                3900 non-null   object 
 4   customer_city                3900 non-null   object 
 5   sales_region                 3900 non-null   object 
 6   sales_channel                3900 non-null   object 
 7   product_category             3900 non-null   object 
 8   product_name                 3900 non-null   object 
 9   brand                        3900 non-null   object 
 10  unit                         3900 non-null   object 
 11  quantity                     3900 non-null   int64  
 12  unit_price_inr               3900 non-null   int64  
 

In [6]:
print("\nStatistical summary:")
print(df.describe(include="all"))


Statistical summary:
       transaction_id transaction_date    customer_id customer_type  \
count            3900             3900           3900          3900   
unique           3900              365            767             3   
top     ORN-TXN-03900       2026-02-14  ORN-CUST-0317        Dealer   
freq                1               25             42          1760   
mean              NaN              NaN            NaN           NaN   
std               NaN              NaN            NaN           NaN   
min               NaN              NaN            NaN           NaN   
25%               NaN              NaN            NaN           NaN   
50%               NaN              NaN            NaN           NaN   
75%               NaN              NaN            NaN           NaN   
max               NaN              NaN            NaN           NaN   

       customer_city   sales_region sales_channel product_category  \
count           3900           3900          3900      

# 3. CHECK DUPLICATE RECORDS

In [7]:
print("\nDuplicate rows:", df.duplicated().sum())

# Remove completely duplicated rows
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)



Duplicate rows: 0
Shape after removing duplicates: (3900, 29)


# 4. CHECK DUPLICATE TRANSACTION IDs


In [9]:
duplicate_transactions = df["transaction_id"].duplicated().sum()

In [10]:
print("\nDuplicate transaction IDs:", duplicate_transactions)


Duplicate transaction IDs: 0


In [11]:
# Keep the first occurrence if duplicate transaction IDs exist
df = df.drop_duplicates(subset="transaction_id", keep="first")

# 5. CHECK MISSING VALUES

In [12]:
missing_values = df.isnull().sum()

print("\nMissing values:")
print(missing_values[missing_values > 0])



Missing values:
discount_pct             39
lead_time_days           18
installation_status    2272
dtype: int64


# 6. HANDLE MISSING VALUES

In [15]:
# Discount percentage:
# Missing discounts are assumed to mean that no discount
# was recorded for the transaction.
df["discount_pct"] = df["discount_pct"].fillna(0)

In [16]:
# Lead time:
# Use the median lead time because it is less affected by
# extreme delivery delays than the mean.
df["lead_time_days"] = df["lead_time_days"].fillna(
    df["lead_time_days"].median()
)

# 7. CONVERT DATA TYPES


In [17]:
# Convert transaction date
df["transaction_date"] = pd.to_datetime(
    df["transaction_date"],
    errors="coerce"
)

In [18]:
# Numerical columns
numeric_columns = [
    "quantity",
    "unit_price_inr",
    "gross_sales_inr",
    "discount_pct",
    "discount_amount_inr",
    "net_sales_inr",
    "gst_amount_inr_synthetic",
    "invoice_value_inr_synthetic",
    "lead_time_days",
    "previous_orders"
]

In [19]:
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 8. CHECK INVALID NUMERICAL VALUES

In [21]:
print("\nInvalid numerical values:")

print("Quantity <= 0:",
      (df["quantity"] <= 0).sum())

print("Discount outside 0-15%:",
      ((df["discount_pct"] < 0) |
       (df["discount_pct"] > 15)).sum())

print("Lead time < 0:",
      (df["lead_time_days"] < 0).sum())


Invalid numerical values:
Quantity <= 0: 0
Discount outside 0-15%: 0
Lead time < 0: 0


# 9. CHECK CATEGORICAL CONSISTENCY

In [22]:

categorical_columns = [
    "customer_type",
    "customer_city",
    "sales_region",
    "sales_channel",
    "product_category",
    "brand",
    "payment_method",
    "payment_status",
    "shipping_type",
    "delivery_status",
    "installation_required",
    "installation_status",
    "order_status"
]

In [23]:
print("\nUnique categorical values:")

for col in categorical_columns:
    print(f"\n{col}:")
    print(df[col].unique())


Unique categorical values:

customer_type:
['Dealer' 'Installer' 'Residential']

customer_city:
['Yavatmal' 'Amravati' 'Nagpur' 'Wardha' 'Nashik' 'Akola' 'Washim'
 'Buldhana' 'Pune' 'Chandrapur']

sales_region:
['Vidarbha Core' 'Maharashtra Extended']

sales_channel:
['B2B Dealer' 'Rooftop Installation' 'Residential Rooftop']

product_category:
['Electrical BOS' 'Mounting Structure' 'Inverter' 'PV Module' 'ACDB/DCDB'
 'Battery']

brand:
['Orion' 'Waaree' 'Adani Solar' 'SF Batteries']

payment_method:
['Credit' 'RTGS' 'Bank Transfer' 'NEFT' 'UPI' 'Credit Card']

payment_status:
['Paid' 'Partial' 'Pending']

shipping_type:
['Express Delivery' 'Scheduled Delivery' 'Pickup' 'Standard Delivery']

delivery_status:
['Delivered' 'Delayed' 'Pending' 'Cancelled']

installation_required:
['No' 'Yes']

installation_status:
[nan 'Scheduled' 'Completed' 'Pending']

order_status:
['Completed' 'Returned' 'Cancelled']


# 10. BUSINESS RULE VALIDATION

In [24]:
# Gross sales should equal quantity × unit price
df["calculated_gross_sales"] = (
    df["quantity"] * df["unit_price_inr"]
)

gross_sales_errors = (
    abs(df["gross_sales_inr"] -
        df["calculated_gross_sales"]) > 1
)

print(
    "\nGross sales calculation errors:",
    gross_sales_errors.sum()
)


Gross sales calculation errors: 0


In [25]:
# Discount amount validation
df["calculated_discount"] = (
    df["gross_sales_inr"] *
    df["discount_pct"] / 100
)

discount_errors = (
    abs(df["discount_amount_inr"] -
        df["calculated_discount"]) > 1
)

print(
    "Discount calculation errors:",
    discount_errors.sum()
)

Discount calculation errors: 35


In [26]:
# Net sales validation
df["calculated_net_sales"] = (
    df["gross_sales_inr"] -
    df["discount_amount_inr"]
)

net_sales_errors = (
    abs(df["net_sales_inr"] -
        df["calculated_net_sales"]) > 1
)

print(
    "Net sales calculation errors:",
    net_sales_errors.sum()
)

Net sales calculation errors: 0


# 11. CHECK LOGICAL BUSINESS RULES


In [27]:
# Cancelled orders should not have Delivered status
invalid_cancelled_orders = (
    (df["order_status"] == "Cancelled") &
    (df["delivery_status"] == "Delivered")
)

print(
    "\nCancelled orders marked Delivered:",
    invalid_cancelled_orders.sum()
)


Cancelled orders marked Delivered: 0


In [28]:
# Installation should be N/A when installation is not required
invalid_installation_status = (
    (df["installation_required"] == "No") &
    (df["installation_status"] != "N/A")
)

print(
    "Invalid installation status:",
    invalid_installation_status.sum()
)

Invalid installation status: 2217


# 12. REMOVE TEMPORARY VALIDATION COLUMN


In [29]:
df.drop(
    columns=[
        "calculated_gross_sales",
        "calculated_discount",
        "calculated_net_sales"
    ],
    inplace=True
)

# 13. OUTLIER DETECTION


In [30]:
# IQR method for numerical business variables

def detect_outliers(data, column):

    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]

    return outliers, lower_bound, upper_bound

In [31]:
for column in [
    "quantity",
    "unit_price_inr",
    "net_sales_inr",
    "lead_time_days"
]:

    outliers, lower, upper = detect_outliers(df, column)

    print(
        f"\n{column}"
        f"\nLower bound: {lower:.2f}"
        f"\nUpper bound: {upper:.2f}"
        f"\nNumber of outliers: {len(outliers)}"
    )


quantity
Lower bound: -11.00
Upper bound: 29.00
Number of outliers: 289

unit_price_inr
Lower bound: -17045.00
Upper bound: 33315.00
Number of outliers: 308

net_sales_inr
Lower bound: -171134.77
Upper bound: 303488.22
Number of outliers: 287

lead_time_days
Lower bound: -2.50
Upper bound: 9.50
Number of outliers: 5


# 15. FINAL DATA QUALITY CHECK

In [32]:
print("FINAL DATA QUALITY CHECK")

FINAL DATA QUALITY CHECK


In [33]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate rows:", df.duplicated().sum())

print(
    "\nDuplicate transaction IDs:",
    df["transaction_id"].duplicated().sum()
)

print("\nData types:")
print(df.dtypes)

Rows: 3900
Columns: 29

Missing values:
installation_status    2272
dtype: int64

Duplicate rows: 0

Duplicate transaction IDs: 0

Data types:
transaction_id                         object
transaction_date               datetime64[ns]
customer_id                            object
customer_type                          object
customer_city                          object
sales_region                           object
sales_channel                          object
product_category                       object
product_name                           object
brand                                  object
unit                                   object
quantity                                int64
unit_price_inr                          int64
gross_sales_inr                       float64
discount_pct                          float64
discount_amount_inr                   float64
net_sales_inr                         float64
gst_amount_inr_synthetic              float64
invoice_value_inr_synthetic  

# 16. SAVE CLEAN DATASET

In [39]:
output_file = "orion_solar_operations_cleaned.csv"

df.to_csv(output_file, index=False)

print("Cleaned dataset created successfully!")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
print(f"File: {output_file}")


# Download to your computer
from google.colab import files

files.download(output_file)

Cleaned dataset created successfully!
Rows: 3900
Columns: 29
File: orion_solar_operations_cleaned.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>